In [ ]:
import pandas as pd
import numpy as np
import os
import math
import openpyxl
from DaySim import DaysimSummary
from Survey import DaysimSummary_Survey

In [ ]:
pd.options.display.float_format = '{:,.1f}'.format

In [3]:
model = DaysimSummary()
survey = DaysimSummary_Survey()

runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...
runVehAvailability = True, loading data...
runWrkSchLocationChoice = True, loading data...
runTripMode = True, loading data...
runTourMode = True, loading data...
runTripDestination = True, loading data...
runTourDestination = True, loading data...
runTripTOD = True, loading data...
runTourTOD = True, loading data...
runDayPattern = True, loading data...


### Helpers

In [4]:
PURPOSE_LABELS = {
    1: "Work",2: "School",3: "Escort",4: "Personal Business",5: "Shop",6: "Meal",7: "SocialRecreation",8: "Workbased"}

PPTYP_LABELS = {
    1: "FT Worker",2: "PT Worker",3: "Retired",4: "Nonworker",5: "Student",6: "Student_Age_16_17",7: "Student_Age_5_15", 8:"Under_Age_5"}

tables = {}

errors = {}

def col_pct(df):
    return df.div(df.sum(axis=0)) * 100

def row_pct(df):
    return df.div(df.sum(axis=1).to_numpy(),axis = 0) * 100

# def compare_summaries(model, survey, pct_func=col_pct):
#     m = pct_func(model)
#     s = pct_func(survey).reindex(m.index, columns=m.columns, fill_value=0)
#     d = m - s #switched
#     return pd.concat([m, s, d], keys=["Model %", "Survey %", "Difference (M-S)"])

def compare_summaries(model, survey, pct_func=col_pct):
    if isinstance(model, pd.DataFrame) and 'n' in survey.columns:
        survey_n = survey['n']
        survey = survey[['psexpfac']]
    else:
        survey_n = None
    m = pct_func(model)
    s = pct_func(survey).reindex(m.index, columns=m.columns, fill_value=0)
    d = m - s
    result = pd.concat([m, s, d], keys=["Model %", "Survey %", "Difference (M-S)"])
    if survey_n is not None:
        result = pd.concat([result, survey_n], keys=["Model %", "Survey %", "Difference (M-S)", "Survey n"])
    return result

def compare_and_store(key, model_df, survey_df, pct_func=col_pct):
    t = compare_summaries(model_df, survey_df, pct_func)
    tables[key] = t
    display(t)


In [5]:
class CompareDaysimSurvey:
    def __init__(self, model, survey):
        self.model = model
        self.survey = survey
        self.tables = {}

    def __getattr__(self, name):
        def wrapper(*args, pct_func=col_pct,**kwargs):
            key = "_".join([name] + [str(a) for a in args])[:31] 
            try:
                 m = getattr(self.model, name)(*args, **kwargs)
                 s = getattr(self.survey, name)(*args, **kwargs)
                 compare_and_store(key, m, s,pct_func=pct_func)
            except Exception as e:
                errors[key] = str(e)
                print(f'Error [{key}]: {e}')
        return wrapper

In [6]:
comparison  = CompareDaysimSurvey(model, survey)

# calibration summaries


In [ ]:
   
    # -- Work Location --------------------------------------------------------
comparison.summary_wrkschloc_ft_nwfh_dist()    

In [ ]:

    # -- School Location ------------------------------------------------------
comparison.summary_wrkschloc_sch_dist() 

In [ ]:

   # -- Parking at work -------------------------------------------------------------
comparison.summary_paid_parking()
       

In [ ]:
     # -- Vehicle availability by driver and vehicle type --------------------------------------------------------
comparison.summary_vehavail_drivers_vehs() 


In [ ]:
# Day patterns - tours or no tours by purpose
compare_and_store("day_pattern_rate_work", model.summary_day_pattern_purpose_rate("work"), survey.summary_day_pattern_purpose_rate("work"))
compare_and_store("day_pattern_rate_school", model.summary_day_pattern_purpose_rate("school"), survey.summary_day_pattern_purpose_rate("school"))
compare_and_store("day_pattern_rate_escort", model.summary_day_pattern_purpose_rate("escort"), survey.summary_day_pattern_purpose_rate("escort"))
compare_and_store("day_pattern_rate_pb", model.summary_day_pattern_purpose_rate("pb"), survey.summary_day_pattern_purpose_rate("pb"))
compare_and_store("day_pattern_rate_shop", model.summary_day_pattern_purpose_rate("shop"), survey.summary_day_pattern_purpose_rate("shop"))
compare_and_store("day_pattern_rate_meal", model.summary_day_pattern_purpose_rate("meal"), survey.summary_day_pattern_purpose_rate("meal"))
compare_and_store("day_pattern_rate_socrec", model.summary_day_pattern_purpose_rate("socrec"), survey.summary_day_pattern_purpose_rate("socrec"))

In [ ]:
# Day patterns - tours by purpose 
compare_and_store("day_pattern_tour_count_work", model.summary_day_pattern_tour_count("work"), survey.summary_day_pattern_tour_count("work"))
compare_and_store("day_pattern_tour_count_school", model.summary_day_pattern_tour_count("school"), survey.summary_day_pattern_tour_count("school"))
compare_and_store("day_pattern_tour_count_escort", model.summary_day_pattern_tour_count("escort"), survey.summary_day_pattern_tour_count("escort"))
compare_and_store("day_pattern_tour_count_pb", model.summary_day_pattern_tour_count("pb"), survey.summary_day_pattern_tour_count("pb"))
compare_and_store("day_pattern_tour_count_shop", model.summary_day_pattern_tour_count("shop"), survey.summary_day_pattern_tour_count("shop"))
compare_and_store("day_pattern_tour_count_meal", model.summary_day_pattern_tour_count("meal"), survey.summary_day_pattern_tour_count("meal"))
compare_and_store("day_pattern_tour_count_socrec", model.summary_day_pattern_tour_count("socrec"), survey.summary_day_pattern_tour_count("socrec"))

In [ ]:
# day patterns - stops and by purpose type
for purpose in ["work", "school", "escort", "pb", "shop", "meal", "socrec"]:
    for suffix in ["Stop","FT","PT","Retired","Nonworker","Stu16","Ch515"]:
        target = f"{purpose}_{suffix}"
        compare_and_store(f"idpd_{target}", 
                          model.summary_ipdp_participation(target),
                          survey.summary_ipdp_participation(target))

In [ ]:
# -- Work from home by income level ------------------------------------------------------
compare_and_store("wfh_by_inc", model.summary_wfh_by_inc(), survey.summary_wfh_by_inc(),pct_func=row_pct)

In [ ]:
  # -- Work-based subtour generation ----------------------------------------
comparison.summary_day_pattern_subtour_purpose_rate()    

In [ ]:
       # -- Work tour mode -------------------------------------------------------
comparison.summary_work_tour_mode()
      
    # -- School tour mode -----------------------------------------------------
comparison.summary_school_tour_mode()
 #
    # -- Escort tour mode -----------------------------------------------------
comparison.summary_escort_tour_mode()

    # -- Other home-based tour mode
comparison.summary_other_home_based_tour_mode()





# -- Trip mode -------------------------------------------------------------
comparison.summary_trip_mode_calib()

In [7]:
# -- Work-based subtour mode ------------------------------------------------------
comparison.summary_work_based_subtour_mode()

psexpfac
                 tourmode                
Model %          Drive Alone         63.0
                 Shared Ride 2       15.8
                 Shared Ride 3+      10.9
                 Walk-Transit         0.2
                 Drive-Transit        0.0
                 Bike                 0.3
                 Walk                 9.8
Survey %         Drive Alone          NaN
                 Shared Ride 2        NaN
                 Shared Ride 3+       NaN
                 Walk-Transit         NaN
                 Drive-Transit        NaN
                 Bike                 NaN
                 Walk                 NaN
Difference (M-S) Drive Alone          NaN
                 Shared Ride 2        NaN
                 Shared Ride 3+       NaN
                 Walk-Transit         NaN
                 Drive-Transit        NaN
                 Bike                 NaN
                 Walk                 NaN

In [ ]:
#  Work and school tour departure and arrival time distributions
comparison.summary_work_tour_arr_tod()   
comparison.summary_work_tour_dep_tod()   
comparison.summary_school_tour_arr_tod()   
comparison.summary_school_tour_dep_tod()  
comparison.summary_school_tour_dep_tod_bin()  
comparison.summary_school_tour_dur()

# Write Output

In [ ]:
rows = []
for key, df in tables.items():
    m = df.loc["Model %"]
    s = df.loc["Survey %"]
    d = df.loc["Difference (M-S)"]

    if(len(m.columns) == 1):
        combined = pd.DataFrame({   
            "dim1_name": m.index.name or "dim1",
            "dim1_value": m.index,
            "dim2_name": pd.NA,
            "dim2_value": pd.NA,
            "Model %": m.iloc[:,0].values,
            "Survey %": s.iloc[:,0].values,
            "Difference (M-S)": d.iloc[:,0].values
        })
    else:
        stacked = (m.stack().rename("Model %")
                   .to_frame()
                   .join(s.stack().rename("Survey %"))
                   .join(d.stack().rename("Difference (M-S)"))
                   .reset_index())
        d1,d2 = stacked.columns[:2]
        combined = stacked.rename(columns={
            d1: "dim1_value",
            d2: "dim2_value"
        })
        combined.insert(combined.columns.get_loc("dim1_value"), "dim1_name", d1)
        combined.insert(combined.columns.get_loc("dim2_value"), "dim2_name", d2)

    combined.insert(0, "comparison_key", key)
    rows.append(combined)

long_df = pd.concat(rows, ignore_index=True)
long_df.to_csv('model_survey_comparisons_2.csv', index=False)

In [ ]:
output_file = "DaySim_Survey_Comparisons.xlsx"
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for sheet_name, df in tables.items():
        df.to_excel(writer, sheet_name=sheet_name[:31])  # Excel sheet names have a max length of 31 characters